In [34]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from option import Option
import itertools

from pricing_methods.black_scholes_analytic import analytic_bs_pricer
from pricing_methods.binomial import binomial_pricer
from pricing_methods.monte_carlo import mc_pricer
from pricing_methods.finite_difference import fd_pricer
from pricing_methods.pinn import pinn_pricer

Testing Against the Analytic Solution

In [35]:
S0_vals = [60, 80, 100, 120, 140]
K = 100
sigma_vals = [0.1, 0.3, 0.6]
r_vals = [0.03, 0.07, 0.0]
div_yield_vals = [0.07, 0.03, 0.0]
T_vals = [0.1, 1.0, 3.0]
type_vals = ["call", "put"]

In [36]:
grid = [Option(S0, K, T, r, sigma, div_yield, option_type, "european")
        for S0, sigma, r, div_yield, T, option_type
        in itertools.product(S0_vals, sigma_vals, r_vals, div_yield_vals,
                             T_vals, type_vals)]

In [37]:
mc_grid = [o for o in grid if o.sigma == 0.3 and o.T == 1.0]
pinn_grid = [o for o in grid if o.sigma == 0.3 and o.T == 1.0
             and o.r == 0.03 and o.option_type == "call"]

In [38]:
def grid_errors(pricer, grid, **kwargs):
    exact = np.array([analytic_bs_pricer(o) for o in grid])
    prices = np.array([pricer(o, **kwargs) for o in grid])
    return prices - exact

In [39]:
bin_errors = grid_errors(binomial_pricer, grid, n_steps=300)
fd_errors = grid_errors(fd_pricer, grid, n_space=300, n_steps=300)
mc_errors = grid_errors(mc_pricer, mc_grid, n_paths=200000, n_steps=100, rng=0)

In [ ]:
# pinn_errors = grid_errors(pinn_pricer, pinn_grid, epochs=5000, n_collocation=5000, n_boundary=500,
#                              n_initial=500, Smin=0.05, Smax=300, resample_rate=100)

KeyboardInterrupt: 

In [40]:
print(np.mean(bin_errors))
print(np.mean(fd_errors))
print(np.mean(mc_errors))

0.003242029332728353
0.009763748393279382
0.005584273245619509


In [ ]:
# print(np.mean(pinn_errors))

Put-Call Parity

In [2]:
S0 = 80.0
K = 100.0
sigma = 0.1
r = 0.07
div_yield = 0.03
T = 1.0

In [3]:
call = Option(S0, K, T, r, sigma, div_yield, "call", "european")
put = Option(S0, K, T, r, sigma, div_yield, "put", "european")

In [4]:
def put_call_parity(C, P, S0, K, T, r, div_yield):
    return np.abs(C - P - (S0 * np.exp(-div_yield * T) - K * np.exp(-r * T)))

In [5]:
analytic_call_price = analytic_bs_pricer(call)
analytic_put_price = analytic_bs_pricer(put)

analytic_error = put_call_parity(analytic_call_price, analytic_put_price, S0, K, T, r, div_yield)

In [6]:
bin_call_price = binomial_pricer(call, n_steps=300)
bin_put_price = binomial_pricer(put, n_steps=300)

bin_error = put_call_parity(bin_call_price, bin_put_price, S0, K, T, r, div_yield)

In [7]:
mc_call_price = mc_pricer(call, n_paths=200000, n_steps=100, rng=0)
mc_put_price = mc_pricer(put, n_paths=200000, n_steps=100, rng=0)

mc_error = put_call_parity(mc_call_price, mc_put_price, S0, K, T, r, div_yield)

In [8]:
fd_call_price = fd_pricer(call, n_space=300, n_steps=300)
fd_put_price = fd_pricer(put, n_space=300, n_steps=300)

fd_error = put_call_parity(fd_call_price, fd_put_price, S0, K, T, r, div_yield)

In [9]:
pinn_call_price = pinn_pricer(call, epochs=5000, n_collocation=5000, n_boundary=500,
                              n_initial=500, Smin=0.05, Smax=300, resample_rate=100)
pinn_put_price = pinn_pricer(put, epochs=5000, n_collocation=5000, n_boundary=500,
                             n_initial=500, Smin=0.05, Smax=300, resample_rate=100)

pinn_error = put_call_parity(pinn_call_price, pinn_put_price, S0, K, T, r, div_yield)

In [10]:
print(analytic_error)
print(bin_error)
print(mc_error)
print(fd_error)
print(pinn_error)

3.552713678800501e-15
1.723066134218243e-13
0.0033217690119808907
0.0029752141965602164
0.3267414126654842


American >= European

In [ ]:
S0 = 80.0
K = 100.0
sigma = 0.1
r = 0.03
div_yield = 0.04
T = 1.0

In [12]:
call_eur = Option(S0, K, T, r, sigma, div_yield, "call", "european")
put_eur = Option(S0, K, T, r, sigma, div_yield, "put", "european")

call_am = Option(S0, K, T, r, sigma, div_yield, "call", "american")
put_am = Option(S0, K, T, r, sigma, div_yield, "put", "american")

In [13]:
bin_call_eur_price = binomial_pricer(call_eur, n_steps=300)
bin_put_eur_price = binomial_pricer(put_eur, n_steps=300)
bin_call_am_price = binomial_pricer(call_am, n_steps=300)
bin_put_am_price = binomial_pricer(put_am, n_steps=300)

print(bin_call_eur_price <= bin_call_am_price)
print(bin_put_eur_price <= bin_put_am_price)

True
True


In [14]:
mc_call_eur_price = mc_pricer(call_eur, n_paths=200000, n_steps=100, rng=0)
mc_put_eur_price = mc_pricer(put_eur, n_paths=200000, n_steps=100, rng=0)
mc_call_am_price = mc_pricer(call_am, n_paths=200000, n_steps=100, rng=0)
mc_put_am_price = mc_pricer(put_am, n_paths=200000, n_steps=100, rng=0)

print(mc_call_eur_price <= mc_call_am_price)
print(mc_put_eur_price <= mc_put_am_price)

True
True


In [15]:
fd_call_eur_price = fd_pricer(call_eur, n_space=300, n_steps=300)
fd_put_eur_price = fd_pricer(put_eur, n_space=300, n_steps=300)
fd_call_am_price = fd_pricer(call_am, n_space=300, n_steps=300)
fd_put_am_price = fd_pricer(put_am, n_space=300, n_steps=300)

print(fd_call_eur_price <= fd_call_am_price)
print(fd_put_eur_price <= fd_put_am_price)

False
True


In [16]:
pinn_call_eur_price = pinn_pricer(call_eur, epochs=5000, n_collocation=5000, n_boundary=500,
                                  n_initial=500, Smin=0.05, Smax=300, resample_rate=100)
pinn_put_eur_price = pinn_pricer(put_eur, epochs=5000, n_collocation=5000, n_boundary=500,
                                 n_initial=500, Smin=0.05, Smax=300, resample_rate=100)
pinn_call_am_price = pinn_pricer(call_am, epochs=5000, n_collocation=5000, n_boundary=500,
                                 n_initial=500, Smin=0.05, Smax=300, resample_rate=100)
pinn_put_am_price = pinn_pricer(put_am, epochs=5000, n_collocation=5000, n_boundary=500,
                                n_initial=500, Smin=0.05, Smax=300, resample_rate=100)

print(pinn_call_eur_price <= pinn_call_am_price)
print(pinn_put_eur_price <= pinn_put_am_price)

False
True


American call with zero dividend yield

In [17]:
S0 = 80.0
K = 100.0
sigma = 0.1
r = 0.07
div_yield = 0.00
T = 1.0

In [18]:
call_eur = Option(S0, K, T, r, sigma, div_yield, "call", "european")
call_am = Option(S0, K, T, r, sigma, div_yield, "call", "american")

In [19]:
bin_call_eur_price = binomial_pricer(call_eur, n_steps=300)
bin_call_am_price = binomial_pricer(call_am, n_steps=300)
bin_call_error = np.abs(bin_call_eur_price - bin_call_am_price)

print(bin_call_error)

0.0


In [20]:
mc_call_eur_price = mc_pricer(call_eur, n_paths=200000, n_steps=100, rng=0)
mc_call_am_price = mc_pricer(call_am, n_paths=200000, n_steps=100, rng=0)
mc_call_error = np.abs(mc_call_eur_price - mc_call_am_price)

print(mc_call_error)

7.487088207897008e-05


In [21]:
fd_call_eur_price = fd_pricer(call_eur, n_space=300, n_steps=300)
fd_call_am_price = fd_pricer(call_am, n_space=300, n_steps=300)
fd_call_error = np.abs(fd_call_eur_price - fd_call_am_price)

print(fd_call_error)

8.382183835919932e-15


In [22]:
pinn_call_eur_price = pinn_pricer(call_eur, epochs=5000, n_collocation=5000, n_boundary=500,
                                  n_initial=500, Smin=0.05, Smax=300, resample_rate=100)
pinn_call_am_price = pinn_pricer(call_am, epochs=5000, n_collocation=5000, n_boundary=500,
                                 n_initial=500, Smin=0.05, Smax=300, resample_rate=100)
pinn_call_error = np.abs(pinn_call_eur_price - pinn_call_am_price)

print(pinn_call_error)

7.210148837417364
